# 機械学習データセットの探索と作成

**学習目標**
* NYC タクシー乗車に関する BigQuery の一般公開データセットにアクセスして探索する
* Seaborn ライブラリを使用してデータセットを可視化する
* 将来の ML モデル トレーニングに向けてデータセットを検査およびクリーンアップする
* 将来の ML モデルのパフォーマンスを評価するためのベンチマークを作成する

## 概要
このノートブックでは、運賃見積もりツールを支援する機械学習モデルを構築するために、ニューヨーク市のタクシー乗車に対応するデータを探索します。乗客が運賃に驚かないように、また請求額が予想よりはるかに高い場合に異議を申し立てることができるように、妥当な運賃を提示することが目的です。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.cloud import bigquery

%load_ext google.cloud.bigquery

<h3> BigQuery からサンプル データを抽出する </h3>

使用するデータセットは、<a href="https://bigquery.cloud.google.com/table/nyc-tlc:yellow.trips">BigQuery の一般公開データセット</a> です。リンクをクリックして、列名を確認してください。[詳細] タブに切り替えてレコード数が 10 億件であることを確認し、[プレビュー] タブに切り替えていくつかの行を確認します。

データセットから関心のあるフィールドを取得する SQL クエリを作成しましょう。タイムスタンプを予測可能な形式で取得することをお勧めします。

In [ ]:
%%bigquery
SELECT
  FORMAT_TIMESTAMP( "%Y-%m-%d %H:%M:%S %Z", pickup_datetime) AS pickup_datetime,
  pickup_longitude,
  pickup_latitude,
  dropoff_longitude,
  dropoff_latitude,
  passenger_count,
  trip_distance,
  tolls_amount,
  fare_amount,
  total_amount
# TODO 1: nyc-tlc yellow タクシー乗車用の正しい BigQuery 一般公開データセットを設定します
# ヒント: ハイフン '-' を含むプロジェクトはバッククォート `` でエスケープしてください
FROM
LIMIT
  10

きれいなグラフを描画できるように、レコード数を増やしてみましょう。レコードが返される順序は保証されていないため、単に LIMIT を増やしただけではどのレコードが返されるかは保証されません。データセットを適切にサンプリングするために、乗車時刻のハッシュ（HASH）を使用して 100,000 件に 1 件のレコードを返すようにしましょう。データには 10 億件のレコードがあるため、これにより約 10,000 件のレコードが返されるはずです。

また、BigQuery のクエリ結果を "trips" という名前の Pandas DataFrame に格納します。

In [ ]:
%%bigquery trips
SELECT
  FORMAT_TIMESTAMP( "%Y-%m-%d %H:%M:%S %Z", pickup_datetime) AS pickup_datetime,
  pickup_longitude,
  pickup_latitude,
  dropoff_longitude,
  dropoff_latitude,
  passenger_count,
  trip_distance,
  tolls_amount,
  fare_amount,
  total_amount
FROM
  `nyc-tlc.yellow.trips`
WHERE
  ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 1

In [ ]:
print(len(trips))

In [ ]:
# Pandas DataFrame は配列のようにスライスできます
trips[:10]

<h3> データの探索 </h3>

このデータセットを探索し、必要に応じてクリーンアップしましょう。グラフの可視化には Python の Seaborn パッケージを、スライスとフィルタリングには Pandas を使用します。

In [ ]:
# TODO 2: Seaborn ライブラリを使用してデータセットを可視化します。
# 乗車距離（distance）を X 軸、運賃（fare amount）を Y 軸としてプロットします。
ax = sns.regplot(
    x="",
    y="",
    fit_reg=False,
    ci=None,
    truncate=True,
    data=trips,
)
ax.figure.set_size_inches(10, 8)

うーん... データに対処が必要な問題があることにお気づきでしょうか。

距離が 0 として記録されている多くの無効なデータや、明らかに不当な運賃がいくつか存在しているようです。これらを分析から除外しましょう。BigQuery クエリを変更して、0 マイルを超える乗車と、最低タクシー運賃（2.50 ドル）以上の運賃のみを保持するようにします。

追加された WHERE 句に注目してください。

In [ ]:
%%bigquery trips
SELECT
  FORMAT_TIMESTAMP( "%Y-%m-%d %H:%M:%S %Z", pickup_datetime) AS pickup_datetime,
  pickup_longitude,
  pickup_latitude,
  dropoff_longitude,
  dropoff_latitude,
  passenger_count,
  trip_distance,
  tolls_amount,
  fare_amount,
  total_amount
FROM
  `nyc-tlc.yellow.trips`
WHERE
  ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 1
    # TODO 3: ゼロ距離以外の乗車と 2.50 ドル以上の運賃のみを含めるようにデータをフィルタリングします
    AND 

In [ ]:
print(len(trips))

In [ ]:
ax = sns.regplot(
    x="trip_distance",
    y="fare_amount",
    fit_reg=False,
    ci=None,
    truncate=True,
    data=trips,
)
ax.figure.set_size_inches(10, 8)

45 ドルと 50 ドル付近にある線（筋）は何でしょうか。これらは JFK 空港やラガーディア空港からマンハッタン内の任意の場所への定額料金の乗車であり、予想されるものです。データを出力して、値が妥当であるかどうかを確認してみましょう。

また、通行料金（toll amount）が合計金額（total amount）に含まれているかどうかも確認しましょう。

In [ ]:
tollrides = trips[trips["tolls_amount"] > 0]
tollrides[tollrides["pickup_datetime"] == "2012-02-27 09:19:10 UTC"]

In [ ]:
notollrides = trips[trips["tolls_amount"] == 0]
notollrides[notollrides["pickup_datetime"] == "2012-02-27 09:19:10 UTC"]

上記のいくつかのサンプルを見ると、合計金額（total_amount）には運賃、通行料、チップがある程度任意に反映されていることがわかります。これは、乗客が現金で支払った場合チップの額が記録されないためです。そのため、予測すべき対象として fare_amount + tolls_amount の合計を使用します。チップは乗客の裁量によるものであり、運賃見積もりツールに含める必要はありません。

また、列内の値の分布も見てみましょう。

In [ ]:
trips.describe()

うーん... 経度（longitude）の最小値と最大値がおかしいようです。

最後に、いくつかの乗車の出発地点と到着地点を実際に見てみましょう。

In [ ]:
def showrides(df, numlines):
    lats = []
    lons = []
    for iter, row in df[:numlines].iterrows():
        lons.append(row["pickup_longitude"])
        lons.append(row["dropoff_longitude"])
        lons.append(None)
        lats.append(row["pickup_latitude"])
        lats.append(row["dropoff_latitude"])
        lats.append(None)

    sns.set_style("darkgrid")
    plt.figure(figsize=(10, 8))
    plt.plot(lons, lats)

In [ ]:
showrides(notollrides, 10)

In [ ]:
showrides(tollrides, 10)

予想どおり、有料道路を利用する乗車は一般的な乗車よりも長距離です。

<h3> 品質管理とその他の前処理 </h3>

データのクリーンアップを行う必要があります。
<ol>
<li>ニューヨーク市の経度はおよそ -74、緯度はおよそ 41 です。</li>
<li>乗客数が 0 の乗車は存在しないはずです。</li>
<li>total_amount 列を fare_amount と tolls_amount のみを反映するようにクリーンアップし、その後これら 2 つの列を削除します。</li>
<li>乗車が始まる前には乗車場所と降車場所はわかりますが、走行距離はわかりません（走行ルートに依存するため）。したがって、走行距離を ML データセットから除外します。</li>
<li>タイムスタンプを破棄します。</li>
</ol>

走行距離 0 の乗車を除外したのと同様に、BigQuery で前処理を行うこともできますが、別の選択肢を示すために Python で実行してみましょう。本番環境では、リアルタイムの入力データに対して同じ前処理を実行する必要があります。 

このような入力データの前処理は、特に品質管理が動的である場合、ML では非常によく行われます。

In [ ]:
def preprocess(trips_in):
    trips = trips_in.copy(deep=True)
    trips.fare_amount = trips.fare_amount + trips.tolls_amount
    del trips["tolls_amount"]
    del trips["total_amount"]
    del trips["trip_distance"]  # 事前にはわかりません。

    qc = np.all(
        [
            trips["pickup_longitude"] > -78,
            trips["pickup_longitude"] < -70,
            trips["dropoff_longitude"] > -78,
            trips["dropoff_longitude"] < -70,
            trips["pickup_latitude"] > 37,
            trips["pickup_latitude"] < 45,
            trips["dropoff_latitude"] > 37,
            trips["dropoff_latitude"] < 45,
            trips["passenger_count"] > 0,
        ],
        axis=0,
    )

    return trips[qc]


tripsqc = preprocess(trips)
tripsqc.describe()

品質管理（QC）により、約 300 行（11,400 - 11,101）またはデータの約 3% が削除されました。これは妥当と思われます。

ML データセットの作成に進みましょう。

<h3> ML データセットの作成 </h3>

QC 済みのデータをトレーニング セット、検証セット、テスト セットにランダムに分割します。
これはデータ全体ではないことに注意してください。データには 10 億件のタクシー乗車記録があります。ここでは小規模なデータセットでの方法を示すために、10,000 件の乗車データを分割しているだけです。実際には 10 億件すべての乗車データに対して行う必要があり、この方法ではスケールしません。

In [ ]:
shuffled = tripsqc.sample(frac=1)
trainsize = int(len(shuffled["fare_amount"]) * 0.70)
validsize = int(len(shuffled["fare_amount"]) * 0.15)

df_train = shuffled.iloc[:trainsize, :]
df_valid = shuffled.iloc[trainsize : (trainsize + validsize), :]  # noqa: E203
df_test = shuffled.iloc[(trainsize + validsize) :, :]  # noqa: E203

In [ ]:
df_train.head(n=1)

In [ ]:
df_train.describe()

In [ ]:
df_valid.describe()

In [ ]:
df_test.describe()

3 つの DataFrame を適切に命名された CSV ファイルに書き出しましょう。すべてのデータで実行する前にコードが動作することを確認するために、これらの CSV ファイルをローカル トレーニングに使用できます（これらのファイルは完全なデータセットのわずか 1/100,000 を表しているにすぎないことを思い出してください）。

In [ ]:
def to_csv(df, filename):
    outdf = df.copy(deep=False)
    outdf.loc[:, "key"] = np.arange(0, len(outdf))  # 行番号をキーとして使用
    # ターゲットが最初の列になるように列の順序を変更
    cols = outdf.columns.tolist()
    cols.remove("fare_amount")
    cols.insert(0, "fare_amount")
    print(cols)  # 新しい列の順序
    outdf = outdf[cols]
    outdf.to_csv(filename, header=False, index_label=False, index=False)


to_csv(df_train, "taxi-train.csv")
to_csv(df_valid, "taxi-valid.csv")
to_csv(df_test, "taxi-test.csv")

In [ ]:
!head -10 taxi-valid.csv

<h3> データセットが存在することの確認 </h3>

In [ ]:
!ls -l *.csv

train、valid、test に対応する 3 つの .csv ファイルが作成されました。ファイルサイズの比率は、データの分割比率に対応しています。

In [ ]:
%%bash
head taxi-train.csv

問題なさそうです。これで ML データセットが準備できたので、ML モデルのトレーニング、検証、評価を行う準備が整いました。

<h3> ベンチマーク </h3>

複雑な ML モデルの構築を始める前に、非常にシンプルなモデルを考案し、それをベンチマークとして使用することをお勧めします。

ここでのモデルは、単に平均運賃（fare_amount）を平均走行距離（trip_distance）で割ってレート（単価）を算出し、それを使用して予測するというものです。このようなモデルの RMSE を計算してみましょう。

In [ ]:
def distance_between(lat1, lon1, lat2, lon2):
    # 直線距離（大圏距離）を計算するためのハバーサインの公式。
    lat1_r = np.radians(lat1)
    lat2_r = np.radians(lat2)
    lon_diff_r = np.radians(lon2 - lon1)
    sin_prod = np.sin(lat1_r) * np.sin(lat2_r)
    cos_prod = np.cos(lat1_r) * np.cos(lat2_r) * np.cos(lon_diff_r)
    minimum = np.minimum(1, sin_prod + cos_prod)
    dist = np.degrees(np.arccos(minimum)) * 60 * 1.515 * 1.609344

    return dist


def estimate_distance(df):
    return distance_between(
        df["pickuplat"], df["pickuplon"], df["dropofflat"], df["dropofflon"]
    )


def compute_rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))


def print_rmse(df, rate, name):
    print(
        "{1} RMSE = {0}".format(
            compute_rmse(df["fare_amount"], rate * estimate_distance(df)), name
        )
    )


# TODO 4: 今後の ML モデルのパフォーマンスを評価するためのベンチマークを作成します

# 5 つの特徴量の列を指定
FEATURES = ["", "", "", "", ""]

# 予測用の 1 つのターゲット列を指定
TARGET = ""

columns = list([TARGET])
columns.append("pickup_datetime")
columns.extend(FEATURES)  # CSV ではターゲットが最初の列で、その後に特徴量が続きます
columns.append("key")
df_train = pd.read_csv("taxi-train.csv", header=None, names=columns)
df_valid = pd.read_csv("taxi-valid.csv", header=None, names=columns)
df_test = pd.read_csv("taxi-test.csv", header=None, names=columns)
rate = df_train["fare_amount"].mean() / estimate_distance(df_train).mean()
print(f"Rate = ${rate}/km")
print_rmse(df_train, rate, "Train")
print_rmse(df_valid, rate, "Valid")
print_rmse(df_test, rate, "Test")

<h2> 同じデータセットでのベンチマーク </h2>

RMSE はデータセットに依存するため、比較のために毎回同じデータセットで評価する必要があります。今後のラボではこのクエリを使用します。

In [ ]:
validation_query = """
SELECT
  (tolls_amount + fare_amount) AS fare_amount,
  pickup_datetime,
  pickup_longitude AS pickuplon,
  pickup_latitude AS pickuplat,
  dropoff_longitude AS dropofflon,
  dropoff_latitude AS dropofflat,
  passenger_count*1.0 AS passengers,
  "unused" AS key
FROM
  `nyc-tlc.yellow.trips`
WHERE
  ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 10000)) = 2
  AND trip_distance > 0
  AND fare_amount >= 2.5
  AND pickup_longitude > -78
  AND pickup_longitude < -70
  AND dropoff_longitude > -78
  AND dropoff_longitude < -70
  AND pickup_latitude > 37
  AND pickup_latitude < 45
  AND dropoff_latitude > 37
  AND dropoff_latitude < 45
  AND passenger_count > 0
"""

client = bigquery.Client()
df_valid = client.query(validation_query).to_dataframe()
print_rmse(df_valid, 2.59988, "Final Validation Set")

単純な距離ベースのルールにより、<b>$8.14</b> の RMSE が得られます。もちろんこれを超える（より小さい RMSE を達成する）必要がありますが、このような単純な経験則を超えることは驚くほど難しいことがわかるでしょう。

しかし、野心的に、テスト セットで RMSE が 6 ドル未満の ML モデルを構築することを目標にしましょう。

Copyright 2020 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.